<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.2
# *Mining Social Media on Reddit*

## The Reddit API and the PRAW Package

The Reddit API is rich and complex, with many endpoints (https://www.reddit.com/dev/api/). It includes methods for navigating its collections, which include various kinds of media as well as comments. Fortunately, the Python library PRAW reduces much of this complexity.

Reddit requires developers to create and authenticate an app before they can use the API, but the process is much less onerous than some, and does not have waiting period for approval of new developers.

### 1. Create a Reddit App

Go to https://www.reddit.com/prefs/apps and click "create an app".

Enter the following in the form:

- a name for your app
- select "script" radio button
- a description
- a redirect URI

(Nb. For pulling data into a data science experiment, a local port can be used for the Redirect URI; try http://127.0.0.1:1410)


- click "create app"
- from the form that displays, copy the following to a local text file (or to this notebook):

  - name (the name you gave to your app)
  - redirect URI
  - personal use script (this is your OAuth 2 Client ID)
  - secret (this is your OAuth 2 Secret)

In [1]:
# Class Exercise

# personal use script:  ASf-kkcz-C39XFg4am6VKQ

# secret:  B36uAPCteG_2jQ-pd3Vvy8fPUdOtrg

# redirect uri:  http://127.0.0.1:1410

### 2. Register for API Access

- follow the link at https://www.reddit.com/wiki/api and read the terms of use for Reddit API access
- fill in the form fields at the bottom
  - make sure to enter your new OAuth Client ID where indicated
  - your use case could be something like "Training in API usage for data science projects"
  - your platform could be something like "Jupyter Notebooks / Python"
  
- click "SUBMIT"

- when asked for User-Agent, enter something that fits this pattern:
  `your_os-python:your_reddit_appname:v1.0 (by /u/your_reddit_username)`

### 3. Load Python Libraries

In [2]:
!pip install praw

In [3]:
import praw
import requests
import json
import pprint
from datetime import datetime, date, time

### 4. Authenticate from your Python script

You could assign your authentication details explicitly, as follows:

In [4]:
my_user_agent = 'Possible_Mousse7711'   # your user Agent string goes in here
my_client_id = 'ASf-kkcz-C39XFg4am6VKQ'   # your Client ID string goes in here
my_client_secret = 'B36uAPCteG_2jQ-pd3Vvy8fPUdOtrg'   # your Secret string goes in here

A better way would be to store these details externally, so they are not displayed in the notebook:

- create a file called "auth_reddit.json" in your "notebooks" directory, and save your credentials there in JSON format:

`{   "my_client_id": "your Client ID string goes in here",` <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;` "my_client_secret": "your Secret string goes in here",` <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`"my_user_agent": "your user Agent string goes in here"` <br>
`}`

Use the following code to load the credentials:  

In [5]:
dictionary = {   "my_client_id": 'ASf-kkcz-C39XFg4am6VKQ',
         "my_client_secret": 'B36uAPCteG_2jQ-pd3Vvy8fPUdOtrg',
        "my_user_agent": 'Possible_Mousse7711'
}

json_object = json.dumps(dictionary, indent=4)

with open('auth_reddit.json', 'w') as outfile:
    outfile.write(json_object)

In [6]:
pwd()  # make sure your working directory is where the file is

'C:\\Users\\shopr\\Downloads'

In [7]:
path_auth = 'auth_reddit.json'
auth = json.loads(open(path_auth).read())
pp = pprint.PrettyPrinter(indent=4)
# For debugging only:

pp.pprint(auth)

{   'my_client_id': 'ASf-kkcz-C39XFg4am6VKQ',
    'my_client_secret': 'B36uAPCteG_2jQ-pd3Vvy8fPUdOtrg',
    'my_user_agent': 'Possible_Mousse7711'}


In [8]:
my_user_agent = auth['my_user_agent']
my_client_id = auth['my_client_id']
my_client_secret = auth['my_client_secret']

Security considerations:
- this method only keeps your credentials invisible as long as nobody else gets access to this notebook file
- if you wanted another user to have access to the executable notebook without divulging your credentials you should set up an OAuth 2.0 workflow to let them obtain and apply their own API tokens when using your app
- if you just want to share your analyses, you could use a separate script (which you don't share) to fetch the data and save it locally, then use a second notebook (with no API access) to load and analyze the locally stored data

### 5. Exploring the API

Here is how to connect to Reddit with read-only access:

In [9]:
reddit = praw.Reddit(client_id = my_client_id,
                     client_secret = my_client_secret,
                     user_agent = my_user_agent)

print('Read-only = ' + str(reddit.read_only))  # Output: True

Read-only = True


In the next cell, put the cursor after the '.' and hit the [tab] key to see the available members and methods in the response object:

In [10]:
reddit.subreddit("python")

Subreddit(display_name='python')

In [11]:
subreddit_name = 'malaysia'
subreddit = reddit.subreddit(subreddit_name)

In [12]:
comments = []
for comment in subreddit.comments(limit=1000):
    comments.append(comment)

In [13]:
reddit.subreddit(subreddit_name)

Subreddit(display_name='malaysia')

Consult the PRAW and Reddit API documentation. Print a few of the response members below:

In [14]:
for top_level_comment in subreddit.comments(limit=3):
    print(top_level_comment.body)

The power grid isn't gonna run out of fuel.
Do you own a drop tail? Nope? Your investment portfolio must be old and not so smart….same for me and everyone else.

If only we all knew how to modernise out investment and do smart businesses…if only..
People became numb to fear mongering titles put there to drive clicks and it's the media's fault for overusing it.


Content in Reddit is grouped by topics called "subreddits". Content, called "submissions", is fetched by calling the `subreddit` method of the connection object (which is our `reddit` variable) with an argument that matches an actual topic.

We also need to append a further method call to a "subinstance", such as one of the following:

- controversial
- gilded
- hot
- new
- rising
- top

One of the submission objects members is `title`. Fetch and print 10 submission titles from the 'learnpython' subreddit using one of the subinstances above:

In [15]:
for submission in reddit.subreddit('learnpython').hot(limit=10):
    print(submission.title)

Ask Anything Monday - Weekly Thread
Ask Anything Monday - Weekly Thread
What do people mean when they say "don't use too many if statements" and how do you avoid it?
Problems getting pywin32 installed
Books to learn python?
Adding comments to code
Return function inside def, called by other def, not returning values. return function doesn't returns value back.
Features worked on localhost but broke on VPS (Python)
What online courses do you think are best for beginners
Need guidance on installing the pyspark in system


Now retrieve 10 authors:

In [16]:
for submission in reddit.subreddit('learnpython').hot(limit=10):
    print(submission.author)

AutoModerator
AutoModerator
Sakuya03692
oakleyguy89
Bananapuddinggggg
cmdwedge75
CommanderRainbowDuck
Quirky-Upstairs-8399
MetalCarnival
YashThebeast


Note that we obtained the titles and authors from separate API calls. Can we expect these to correspond to the same submissions? If not, how could we gurantee that they do?

In [17]:
submissions=reddit.subreddit('learnpython').hot(limit=10)
for submission in submissions:
    print("Author: {} | Title: {}".format(submission.author, submission.title))

Author: AutoModerator | Title: Ask Anything Monday - Weekly Thread
Author: AutoModerator | Title: Ask Anything Monday - Weekly Thread
Author: Sakuya03692 | Title: What do people mean when they say "don't use too many if statements" and how do you avoid it?
Author: oakleyguy89 | Title: Problems getting pywin32 installed
Author: Bananapuddinggggg | Title: Books to learn python?
Author: cmdwedge75 | Title: Adding comments to code
Author: CommanderRainbowDuck | Title: Return function inside def, called by other def, not returning values. return function doesn't returns value back.
Author: Quirky-Upstairs-8399 | Title: Features worked on localhost but broke on VPS (Python)
Author: MetalCarnival | Title: What online courses do you think are best for beginners
Author: YashThebeast | Title: Need guidance on installing the pyspark in system


Why doesn't the next cell produce output?

In [18]:
for submission in submissions:
    print(submission.comments)

Print two comments associated with each of these submissions:

In [19]:
submissions = reddit.subreddit('learnpython').hot(limit=10)
for submission in submissions:
    top_level_comments = list(submission.comments)
    all_comments = submission.comments.list()[:2]
    for comment in all_comments:
        print(comment.body)

Is it really important to master the fundamentals before proceeding to frameworks then build projects? Or just learn enough of the basics and do simple projects?
[removed]
I am using Spyder with Anaconda in Windows for some basic scripting, more complicated stuff I'll do in Ubuntu but that's not relevant here.

When I try to plot something with matplotlib, sometimes it plots in Spyder's "Plots" tab and everything is fine. Sometimes it tries to open a new window that immediately freezes and crashes the kernel. Obviously I would prefer only the first to occur, but I don't know what leads to one or the other. Has anyone encountered this before or knows how to fix it?

>Python 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
Type "copyright", "credits" or "license" for more information.

>IPython 8.27.0 -- An enhanced Interactive Python.



Edit: when other people have this problem it's [really fucky](https://discourse.matplotlib.org/t/windows

Referring to the API documentation, explore the submissions object and print some interesting data:

#### Posting to Reddit

To be able to post to your Reddit account (i.e. contribute submissions), you need to connect to the API with read/write privilege. This requires an *authorized instance*, which is obtained by including your Reddit user name and password in the connection request:

In [20]:
reddit = praw.Reddit(client_id='my client id',
                     client_secret='my client secret',
                     user_agent='my user agent',
                     username='my username',
                     password='my password')
print(reddit.read_only)  # Output: False

False


You could hide these last two credentials by adding them to your JSON file and then reading all five values at once.

>
>


>
>




---



---



> > > > > > > > > © 2025 Institute of Data


---



---



